# CURB - Enforcement Prioritizer
### notebook 02 - Problem Statement 1 (Parking-Induced Congestion)

Turns the ranked hotspots into a **control-room-facing priority list** and emits
frontend-ready files (`outputs/curb_priorities.json` + `.csv`).

**Honest framing (read once):**
- `congestion_impact_index` is the severity x footprint score **normalised to
  0-100** - an INDEX, not a measured or predicted % traffic change. There is no
  congestion measurement in this dataset.
- We do **not** assume how many units are available. Each spot is described by
  its **own measured enforcement history**; deployment is the control room's call.
- Nothing about the force is hardcoded. Citywide staffing is shown as a
  day-by-day **spread** (descriptive context), never used as a fixed capacity.

Per spot, the list stays simple - rank, impact index, a **Chronic / Frequent /
Occasional** tag, cause -> fix, one deploy note - while richer fields stay in the
data for the frontend.

In [ ]:
# --- setup ---
import os, sys, json
from pathlib import Path
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config.py").exists() and (REPO_ROOT.parent / "config.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT)); os.chdir(REPO_ROOT)

import config, pandas as pd, matplotlib.pyplot as plt
from src.pipeline import run as run_pipeline
from src.prioritizer import prioritize, export, citywide_context
DATA_PATH = config.DATA_PATH
pd.set_option("display.max_colwidth", 38)
print("data:", DATA_PATH, "exists:", os.path.exists(DATA_PATH))

## 1 - Build hotspots, then prioritize

In [ ]:
hot = run_pipeline(DATA_PATH, write=False)        # cluster + impact + root cause
context = citywide_context(DATA_PATH)             # descriptive staffing spread
prio = prioritize(hot, DATA_PATH)                 # add index + persistence + notes
print(f"{len(prio):,} prioritized hotspots")

## 2 - The clean list view (what an officer reads)

In [ ]:
prio.head(15)[["rank", "name", "congestion_impact_index",
               "persistence_tier", "root_cause", "deploy_note"]]

## 3 - The full field set (what the frontend receives)

In [ ]:
from src.prioritizer import FRONTEND_FIELDS
prio.head(8)[[c for c in FRONTEND_FIELDS if c in prio.columns]]

## 4 - Persistence triage + impact-coverage curve

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

order = ["Chronic", "Frequent", "Occasional"]
prio["persistence_tier"].value_counts().reindex(order).plot.bar(
    ax=ax[0], color=["#ef4444", "#f59e0b", "#94a3b8"])
ax[0].set_title("Persistence triage"); ax[0].set_ylabel("hotspots")
for i, v in enumerate(prio["persistence_tier"].value_counts().reindex(order)):
    ax[0].text(i, v, f"{v:,}", ha="center", va="bottom")

ax[1].plot(range(1, len(prio) + 1), prio["cumulative_coverage_pct"], color="#3b82f6")
ax[1].set_xlim(0, 500); ax[1].set_xlabel("hotspots addressed (priority order)")
ax[1].set_ylabel("% of total obstruction impact")
ax[1].set_title("Cumulative impact coverage"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

for n in (50, 100, 250):
    print(f"top {n:>3} spots -> {prio.iloc[n-1]['cumulative_coverage_pct']}% of total obstruction impact")

## 5 - Priority view per root cause\nThe *fix* differs by cause, so a planner can filter to the lever they control.

In [ ]:
for cause in ["Delivery overflow", "Safety risk", "Commuter overflow", "Hire / transit demand"]:
    sub = prio[prio.root_cause == cause].head(3)
    if len(sub):
        print(f"\n### {cause}  ->  {sub.iloc[0]['recommended_fix']}")
        print(sub[["rank", "name", "congestion_impact_index", "persistence_tier"]].to_string(index=False))

## 6 - Citywide staffing context (descriptive only)
Shown as a **day-by-day spread**, not a single number - daily staffing varies and
is never used as an input to the ranking.

In [ ]:
# day-by-day distinct officers (force-on-street), straight from the data
src_cols = ["latitude", "longitude", "created_by_id", "created_datetime", "validation_status"]
d = pd.read_csv(DATA_PATH, usecols=src_cols)
d = d[~d.validation_status.isin(config.DROP_VALIDATION)].dropna(subset=["latitude", "longitude"])
b = config.BBOX
d = d[d.latitude.between(b["lat_min"], b["lat_max"]) & d.longitude.between(b["lon_min"], b["lon_max"])]
day = pd.to_datetime(d.created_datetime, errors="coerce", utc=True).dt.tz_convert("Asia/Kolkata").dt.date
opd = d.groupby(day).created_by_id.nunique()

opd.plot(figsize=(11, 3), color="#10b981")
plt.title("Distinct officers active per day (descriptive context)")
plt.ylabel("officers"); plt.tight_layout(); plt.show()
print(json.dumps(context, indent=2))

## 7 - Export for the frontend

In [ ]:
csv_path, json_path = export(prio, context)
print("wrote:", csv_path)
print("wrote:", json_path)

# peek at one frontend record
sample = json.load(open(json_path))
print("\nmeta:", json.dumps(sample["meta"], indent=2))
print("\nfirst hotspot record:")
print(json.dumps(sample["hotspots"][1], indent=2))

## Methodology note (for the deck / when a judge asks)
- **Ranking** is by an obstruction-impact index (severity x vehicle footprint),
  not measured traffic delay. We claim *priority*, never a traffic-% outcome.
- **Persistence** (Chronic/Frequent/Occasional) is measured from each spot's own
  active-day history in the data - the one operational signal the data supports
  cleanly. Thresholds are documented constants in `src/prioritizer.py`.
- **No assumed force size.** `units_typical`/`units_peak` are each spot's measured
  history; citywide staffing is descriptive context only.
- Everything recomputes from whatever data is loaded; nothing is hardcoded except
  the documented severity/footprint/persistence constants.